# Live ALPSS box plot

In [1]:
# --- config ---
DATA_TYPE = 'pdv_alpss_result'

# What to plot. 'Velocity at Max Compression' is present on ~100% of results;
# 'Spall Strength' on only ~7%, and the most recent experiments often have none --
# which is why a spall view can look empty while shots are landing.
METRIC = 'Velocity at Max Compression (m/s)'

# Seeding is driven by IGSN coverage of the chosen METRIC, not by an item budget:
# with velocity ~400 items fill 12 IGSNs; with spall it takes ~1200.
SEED_IGSNS = 12
MAX_SEED_ITEMS = 1500    # hard ceiling; one HTTP GET per item

POLL_SECONDS = 30
WINDOW_SIZE = 10         # boxes visible at once
MIN_BOX_N = 5            # below this, draw points only -- a box over n=2 is noise
HIGHLIGHT_LAST_K = 12    # rows counted as "new"
IQR_MULT = 3.0           # group-wise outlier trim, as in alpss_data_modeling
JITTER = 0.18
MAX_WORKERS = 12

# Replay mode: hold back the newest rows and feed them in, to exercise the live
# path without waiting on the lab. "New" is an arrival counter, not a clock, so
# this drives the exact same highlight code the live path runs.
SIMULATE = False
SIM_HOLDBACK = 40
SIM_ROWS_PER_TICK = 4

In [2]:
import asyncio
import io
import sys
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import ipywidgets as W
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from dotenv import find_dotenv, load_dotenv
from IPython.display import display

load_dotenv(find_dotenv())
sys.path.insert(0, str(Path.cwd().parent / 'src'))

from aimdl_examples import get_girder_client, get_item_to_memory
from aimdl_examples.alpss import ALPSS_MATERIAL_PROPS_FORM_ID

session = requests.Session()
gc = get_girder_client(session=session)
print('connected:', gc.urlBase)

connected: https://data.htmdec.org/api/v1/


In [3]:
# --- fetch + parse ---
UNKNOWN = 'unknown'

SKIP = object()   # parsed fine, nothing to plot (permanent) -- distinct from None (error)

# Whatever ALPSS reports gets plotted. A rejected analysis reports no value, and a row
# with no value for the selected metric simply is not in the view. No quality flags.
METRICS = {
    'Velocity at Max Compression (m/s)': {
        'src': 'Velocity at Max Compression', 'scale': 1.0,
        'ylabel': 'Velocity at max compression (m/s)'},
    'Spall Strength (GPa)': {
        'src': 'Spall Strength', 'scale': 1e-9,
        'ylabel': 'Spall strength (GPa)'},
}


def parse_experiment_ts(v):
    """meta.experiment_date -> tz-aware UTC Timestamp.

    Three clocks exist on a result and only this one is the experiment:
      item.created          -> when the result was INGESTED
      CSV 'Date' + 'Time'   -> when ALPSS dagster RAN
      meta.experiment_date  -> when the shot was FIRED   <- this one

    format='ISO8601' is load-bearing. experiment_date is day-granularity today
    ('2026-07-06') and becomes full ISO ('2026-07-06T21:45:25+00:00') once the
    upstream bug is fixed. Plain to_datetime(utc=True) infers the format from the
    FIRST value it sees and silently coerces the other form to NaT -- so on the day
    the fix lands mid-corpus, every fixed timestamp would vanish from every window
    with no error. utc=True also reconciles the tz-naive date-only form against the
    tz-aware fixed form, which otherwise raises on comparison.
    """
    if v is None or v == '':
        return pd.NaT   # to_datetime(None) returns None, not NaT
    return pd.to_datetime(v, utc=True, errors='coerce', format='ISO8601')


def poll_count(gc):
    """Cheap change-detector: one request, no downloads."""
    return gc.get('aimdl/count').get(DATA_TYPE)


def page_items(gc, offset, limit=50):
    return gc.get('aimdl/datafiles', parameters={
        'dataType': DATA_TYPE, 'limit': limit, 'offset': offset,
        'sort': 'created', 'sortdir': -1,
        'extraFields': '["meta.experiment_date"]',
    })


def fetch_new_items(gc, seen_ids):
    """Walk aimdl/datafiles newest-first, stop at the first already-examined itemId.

    aimdl/datafiles has no `since` param (only aimdl/partition does), and `created` is
    server-assigned at ingest, so newest-first + stop-on-seen is the incremental fetch.
    This is why the library's paginate_datafiles is not reused: it walks every item and
    issues one GET per result -- 3.7k downloads per pass.

    `seen_ids` must contain every item ever EXAMINED, not just the ones that yielded a
    row -- otherwise every poll re-walks and re-downloads the items it already rejected.
    """
    out, offset = [], 0
    while True:
        batch = page_items(gc, offset)
        if not batch:
            return out
        for it in batch:
            if it['_id'] in seen_ids:
                return out
            out.append(it)
        if len(batch) < 50:
            return out
        offset += 50


def parse_row(gc, item):
    """item -> row dict | SKIP (no metric data, permanent) | None (error, retry later).

    Never raises: one bad CSV must not kill a long-running loop.
    """
    try:
        buf = io.BytesIO(b''.join(get_item_to_memory(gc, item)))
        buf.seek(0)
        df = pd.read_csv(buf)
        if df.empty:
            return SKIP
        r = df.iloc[0]
        meta = item.get('meta', {})
        row = {
            'itemId': item['_id'],
            'igsn': meta.get('igsn'),
            'created': pd.to_datetime(item['created'], utc=True, errors='coerce', format='ISO8601'),
            'experiment_ts': parse_experiment_ts(meta.get('experiment_date')),
        }
        found = False
        for name, cfg in METRICS.items():
            v = pd.to_numeric(r.get(cfg['src']), errors='coerce')
            row[name] = float(v) * cfg['scale'] if pd.notna(v) else np.nan
            found |= pd.notna(v)
        if not found:
            return SKIP   # no metric at all -> nothing to plot, ever
        return row
    except Exception as e:
        print(f'  retry {item.get("_id")}: {type(e).__name__}: {e}')
        return None


def parse_many(gc, items):
    if not items:
        return []
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
        return list(ex.map(lambda it: parse_row(gc, it), items))


def build_material_map(gc):
    """base IGSN -> materialType. Missing for a large fraction of samples."""
    entries = gc.get('entry', parameters={'formId': ALPSS_MATERIAL_PROPS_FORM_ID})
    out = {}
    for e in entries:
        d = e.get('data', {})
        if d.get('IGSN'):
            out[d['IGSN']] = d.get('materialType')
    return out


def material_for(igsn, mmap):
    # base-IGSN split mirrors enrich_alpss_with_material_properties (alpss.py:166)
    return mmap.get(str(igsn).split('-')[0]) or UNKNOWN

In [4]:
# --- state ---
class Store:
    """Accumulates rows across polls.

    `seen` holds every EXAMINED itemId (including ones with no spall data), so the
    incremental fetch stops immediately when nothing has changed.
    `arrival_seq` defines "new". Jitter is stamped once, at admission.
    """

    def __init__(self, seed=10):
        self.rows, self.seen, self.seq = [], set(), 0
        self.rng = np.random.default_rng(seed)

    def _admit(self, r):
        self.seen.add(r['itemId'])
        self.seq += 1
        r['arrival_seq'] = self.seq
        # stamped once and never recomputed -- jitter recomputed per redraw makes
        # every point twitch on every frame
        r['_jit'] = float(self.rng.uniform(-JITTER, JITTER))
        self.rows.append(r)

    def ingest(self, items, results):
        """-> (added, skipped, errors). Errors are NOT marked seen, so they retry."""
        added = skipped = errors = 0
        for it, r in zip(items, results):
            if it['_id'] in self.seen:
                continue
            if r is None:            # transient: leave unseen for the next poll
                errors += 1
            elif r is SKIP:          # permanent: mark seen, never download again
                self.seen.add(it['_id'])
                skipped += 1
            else:
                self._admit(r)
                added += 1
        return added, skipped, errors

    def add(self, rows):
        """Admit already-parsed rows (replay path)."""
        n = 0
        for r in rows:
            if r and r is not SKIP and r['itemId'] not in self.seen:
                self._admit(r)
                n += 1
        return n

    def frame(self):
        return pd.DataFrame(self.rows) if self.rows else pd.DataFrame()

    def igsns(self, metric=None):
        """IGSNs with at least one row, optionally restricted to rows having `metric`."""
        return {r['igsn'] for r in self.rows
                if metric is None or pd.notna(r.get(metric))}


store = Store()
mmap = build_material_map(gc)
print(f'material map: {len(mmap)} base IGSNs')

material map: 8 base IGSNs


In [5]:
# --- seed (coverage-driven: page until SEED_IGSNS IGSNs have spall rows) ---
_pending_rows = []
_last_count = None


def seed(target_igsns=SEED_IGSNS, max_items=MAX_SEED_ITEMS):
    """Page newest-first until SEED_IGSNS IGSNs have spall rows, then ingest OLDEST-first.

    The two orders are deliberate. Paging must go newest-first (that is the only order
    aimdl/datafiles offers, and we want the most recent IGSNs). Ingestion must go
    oldest-first, because arrival_seq is what "new" means -- ingest in page order and
    the highest arrival_seq lands on the OLDEST row, so the highlight marks exactly the
    wrong points.
    """
    items_all, rows_all = [], []
    offset = scanned = 0
    while scanned < max_items:
        batch = page_items(gc, offset)
        if not batch:
            break
        rows = parse_many(gc, batch)
        items_all += batch
        rows_all += rows
        scanned += len(batch)
        offset += len(batch)
        usable = {r['igsn'] for r in rows_all
                  if r and r is not SKIP and pd.notna(r.get(METRIC))}
        print(f'  scanned {scanned:5} | rows {sum(1 for r in rows_all if r and r is not SKIP):4} | '
              f'IGSNs w/ metric {len(usable):3}')
        if len(usable) >= target_igsns:
            break

    added, skipped, errors = store.ingest(list(reversed(items_all)), list(reversed(rows_all)))
    print(f'  ingested +{added} | {skipped} without any metric | {errors} errors')
    return scanned


scanned = seed()
_last_count = poll_count(gc)
print(f'seeded {len(store.rows)} rows / {len(store.igsns(METRIC))} IGSNs with {METRIC} '
      f'from {scanned} items | server count={_last_count}')

if SIMULATE:
    # hold back the newest rows; un-see them so the replay re-admits them
    held = sorted(store.rows, key=lambda r: r['arrival_seq'])[-SIM_HOLDBACK:]
    keep = {id(r) for r in held}
    store.rows = [r for r in store.rows if id(r) not in keep]
    for r in held:
        store.seen.discard(r['itemId'])
    _pending_rows = held
    print(f'holding back {len(_pending_rows)} rows for replay')


def _sim_feed():
    global _pending_rows
    if not _pending_rows:
        return 'replay exhausted'
    batch, _pending_rows = _pending_rows[:SIM_ROWS_PER_TICK], _pending_rows[SIM_ROWS_PER_TICK:]
    return f'replayed +{store.add(batch)} ({len(_pending_rows)} left)'

  scanned    50 | rows   50 | IGSNs w/ metric   6
  scanned   100 | rows  100 | IGSNs w/ metric   9
  scanned   150 | rows  150 | IGSNs w/ metric   9
  scanned   200 | rows  200 | IGSNs w/ metric   9
  scanned   250 | rows  250 | IGSNs w/ metric  10
  scanned   300 | rows  300 | IGSNs w/ metric  10
  scanned   350 | rows  350 | IGSNs w/ metric  11
  scanned   400 | rows  400 | IGSNs w/ metric  13
  ingested +400 | 0 without any metric | 0 errors
seeded 400 rows / 13 IGSNs with Velocity at Max Compression (m/s) from 400 items | server count=3695


In [6]:
# --- view ---
TIMEFRAMES = {
    'all': None,
    'last hour': pd.Timedelta(hours=1),
    'last 24h': pd.Timedelta(days=1),
    'last 7d': pd.Timedelta(days=7),
    'last 30d': pd.Timedelta(days=30),
}


def metric_base(metric):
    """Every stored row that reports a value for this metric."""
    df = store.frame()
    if df.empty:
        return df
    df = df[df[metric].notna()].copy()
    df['material_type'] = [material_for(i, mmap) for i in df['igsn']]
    return df


def apply_timeframe(df, tf):
    if tf == 'all' or df.empty:
        return df
    cutoff = pd.Timestamp.now(tz='UTC') - TIMEFRAMES[tf]
    return df[df['experiment_ts'].notna() & (df['experiment_ts'] >= cutoff)]


def iqr_filter(df, xcol, ycol, mult):
    """Group-wise 3*IQR trim, skipped for groups too small to have a meaningful IQR."""
    if df.empty:
        return df
    g = df.groupby(xcol)[ycol]
    q1, q3 = g.transform(lambda x: x.quantile(0.25)), g.transform(lambda x: x.quantile(0.75))
    iqr = q3 - q1
    small = g.transform('size') < MIN_BOX_N
    return df[small | ((df[ycol] >= q1 - mult * iqr) & (df[ycol] <= q3 + mult * iqr))]


def ordered_categories(df, xcol):
    """Oldest -> newest. experiment_ts is day-granular today, so every shot fired on one
    day ties; `created` breaks the tie with ingestion order. The tiebreak simply stops
    mattering once experiment_date carries a time."""
    if df.empty:
        return []
    key = df.groupby(xcol).agg(ts=('experiment_ts', 'max'), cr=('created', 'max'))
    return key.sort_values(['ts', 'cr'], na_position='first').index.tolist()


def build_view(metric, xcol, tf, win_size, win_off):
    """-> (view, cats, stats, base). `base` is the metric-filtered store, used for the
    stable y-axis and for deciding which arrivals count as new."""
    base = metric_base(metric)
    if base.empty:
        return base, [], {'n_all': 0, 'n_no_ts': 0, 'n_cats': 0}, base

    stats = {'n_all': len(base), 'n_no_ts': int(base['experiment_ts'].isna().sum())}
    df_tf = iqr_filter(apply_timeframe(base, tf), xcol, metric, IQR_MULT)

    cats_all = ordered_categories(df_tf, xcol)
    stats['n_cats'] = len(cats_all)
    hi = max(0, len(cats_all) - win_off)
    lo = max(0, hi - win_size)
    cats = cats_all[lo:hi]
    return df_tf[df_tf[xcol].isin(cats)], cats, stats, base


def new_item_ids(base, k):
    """Newest arrivals AMONG PLOTTABLE ROWS -- otherwise the badge would count rows that
    could never appear under the current metric."""
    if base.empty or k <= 0:
        return set()
    return set(base.nlargest(min(k, len(base)), 'arrival_seq')['itemId'])

In [7]:
# --- render ---
_PALETTE = plt.get_cmap('tab10').colors
_GRAY = (0.78, 0.78, 0.78)


def material_colors(mats):
    known = sorted({m for m in mats if m and m != UNKNOWN})
    cmap = {m: _PALETTE[i % len(_PALETTE)] for i, m in enumerate(known)}
    cmap[UNKNOWN] = _GRAY
    return cmap


def stable_ylim(series):
    """Axis fixed off the whole metric-filtered store, not the window -- a y-axis that
    rescales every tick makes the animation unreadable. Quantiles so one outlier cannot
    flatten it. Velocity goes negative on bad fits, so the floor is not pinned to 0."""
    s = series.dropna()
    if s.empty:
        return 0.0, 1.0
    lo, hi = float(s.quantile(0.005)), float(s.quantile(0.995))
    lo = min(0.0, lo)
    if hi <= lo:
        hi = lo + 1.0
    pad = 0.05 * (hi - lo)
    return lo - (pad if lo < 0 else 0.0), hi + pad


def render(ax, view, cats, xcol, metric, base, new_ids, highlight):
    ax.clear()
    pos = {c: i for i, c in enumerate(cats)}
    empty = view.empty
    cmap = material_colors(view['material_type']) if not empty else {UNKNOWN: _GRAY}

    box_data, box_pos, box_mat = [], [], []
    for c in cats:
        sub = view[view[xcol] == c]
        vals = sub[metric].dropna().values
        if len(vals) >= MIN_BOX_N:
            box_data.append(vals)
            box_pos.append(pos[c])
            box_mat.append(c if xcol == 'material_type' else sub['material_type'].iloc[0])

    if box_data:
        bp = ax.boxplot(
            box_data, positions=box_pos, widths=0.65, showfliers=False, patch_artist=True,
            medianprops=dict(color='black', linewidth=1.2),
            whiskerprops=dict(color='black', linewidth=0.8),
            capprops=dict(color='black', linewidth=0.8),
        )
        for patch, mat in zip(bp['boxes'], box_mat):
            patch.set_facecolor(cmap.get(mat, _GRAY))
            patch.set_alpha(0.35)
            patch.set_edgecolor('black')
            patch.set_linewidth(0.8)

    if not empty:
        xs = view[xcol].map(pos).astype(float) + view['_jit']
        is_new = view['itemId'].isin(new_ids) & bool(highlight)
        old = ~is_new
        ax.scatter(xs[old], view.loc[old, metric], color='black', s=9, alpha=0.4,
                   zorder=10, clip_on=False)
        if is_new.any():
            ax.scatter(xs[is_new], view.loc[is_new, metric], color='#d62728', s=40,
                       alpha=0.95, zorder=12, edgecolor='white', linewidth=0.6, clip_on=False)
    else:
        # A timeframe can hold plenty of items yet no rows reporting this metric --
        # spall exists on only ~7% of results. Say which, rather than a blank axis.
        newest = base['experiment_ts'].max() if not base.empty else pd.NaT
        msg = f'no {metric} results in this timeframe'
        if pd.notna(newest):
            msg += f'\nnewest: {newest.date()}'
        ax.text(0.5, 0.5, msg, ha='center', va='center', transform=ax.transAxes,
                fontsize=10, color='#777')

    counts = view[xcol].value_counts().to_dict() if not empty else {}
    ax.set_xticks(np.arange(len(cats)))
    ax.set_xticklabels([f'{c}\n(n={counts.get(c, 0)})' for c in cats], rotation=45, ha='right')
    ax.set_xlim(-0.5, max(len(cats) - 0.5, 0.5))
    ax.set_ylim(*stable_ylim(base[metric] if not base.empty else pd.Series(dtype=float)))
    ax.set_xlabel({'igsn': 'IGSN', 'material_type': 'Material'}[xcol])
    ax.set_ylabel(METRICS[metric]['ylabel'])
    ax.axhline(0, color='#999', linewidth=0.6, zorder=1)
    ax.grid(True, linestyle='--', alpha=0.25, zorder=0)

    if xcol == 'igsn' and not empty:
        present = list(dict.fromkeys(view['material_type']))
        handles = [plt.Line2D([], [], marker='s', linestyle='', markersize=8,
                              markerfacecolor=cmap[m], markeredgecolor='black',
                              alpha=0.6, label=m) for m in present]
        ax.legend(handles=handles, fontsize=7, loc='upper left', frameon=False, ncol=2)

In [8]:
# --- controls ---
w_metric = W.Dropdown(options=list(METRICS), value=METRIC, description='Metric')
w_highlight = W.Checkbox(value=True, description='Highlight new')
w_tf = W.Dropdown(options=list(TIMEFRAMES), value='last 24h', description='Timeframe')
w_x = W.Dropdown(options=['igsn', 'material_type'], value='igsn', description='x-axis')
w_size = W.IntSlider(value=WINDOW_SIZE, min=3, max=25, description='Boxes')
w_follow = W.Checkbox(value=True, description='Follow live')
w_poll = W.BoundedIntText(value=POLL_SECONDS, min=5, max=600, description='Poll (s)')
b_prev = W.Button(description='\u25c0 Older')
b_next = W.Button(description='Newer \u25b6')
b_stop = W.Button(description='Stop', button_style='danger')
status = W.HTML()

# An Image widget, NOT an Output widget. `with out: display(fig)` only captures while a
# cell is executing -- the async poll loop has no cell context, so every tick's figure
# escapes the widget and stacks in the cell's output area instead. Assigning
# Image.value is a comm message, so it lands in the widget from any context.
canvas = W.Image(format='png', layout=W.Layout(width='880px'))

_win_off = 0
fig, ax = plt.subplots(figsize=(max(6.0, 0.9 * WINDOW_SIZE), 4.2))
fig.subplots_adjust(left=0.10, right=0.98, bottom=0.34, top=0.92)
plt.close(fig)   # never auto-shown; it only ever leaves via savefig below


def redraw(msg=''):
    global _win_off
    if w_follow.value:
        _win_off = 0
    metric = w_metric.value
    view, cats, stats, base = build_view(metric, w_x.value, w_tf.value, w_size.value, _win_off)
    new_ids = new_item_ids(base, HIGHLIGHT_LAST_K) if w_highlight.value else set()
    render(ax, view, cats, w_x.value, metric, base, new_ids, w_highlight.value)

    shown = set(view['itemId']) if not view.empty else set()
    outside = len(new_ids - shown)
    ax.set_title(f'{stats["n_all"]} plottable | {stats["n_cats"]} {w_x.value} in timeframe '
                 f'| showing {len(cats)}', pad=8)

    bits = [msg]
    if outside:
        # highlight keys on arrival, the window filters on experiment time -- a result
        # ingested now can carry an old experiment_date. Say so; never silently drop.
        bits.append(f'<b style="color:#d62728">{outside} new outside window</b>')
    if stats['n_no_ts']:
        bits.append(f'{stats["n_no_ts"]} rows lack experiment_date')
    if _win_off:
        bits.append(f'scrolled back {_win_off}')
    status.value = ' &nbsp;|&nbsp; '.join(b for b in bits if b)

    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=96, bbox_inches='tight')
    canvas.value = buf.getvalue()


def _shift(delta):
    def handler(_):
        global _win_off
        w_follow.value = False
        _, _, stats, _ = build_view(w_metric.value, w_x.value, w_tf.value, w_size.value, 0)
        _win_off = int(np.clip(_win_off + delta, 0, max(0, stats['n_cats'] - w_size.value)))
        redraw()
    return handler


b_prev.on_click(_shift(+1))
b_next.on_click(_shift(-1))
for _w in (w_metric, w_highlight, w_tf, w_x, w_size, w_follow):
    _w.observe(lambda ch: redraw(), names='value')

display(W.VBox([
    W.HBox([w_metric, w_highlight, w_tf, w_x]),
    W.HBox([w_size, b_prev, b_next, w_follow]),
    W.HBox([w_poll, b_stop]),
    status, canvas,
]))
redraw('seeded')

In [9]:
# --- live loop ---
# async, not `while True: time.sleep()`: a blocking sleep starves the kernel event loop
# and the widgets above go dead until it exits. Network work is pushed to an executor
# for the same reason.
_running = False


async def _tick():
    global _last_count
    if SIMULATE:
        return _sim_feed()

    loop = asyncio.get_event_loop()
    count = await loop.run_in_executor(None, poll_count, gc)
    if count == _last_count:
        return f'no change (count={count})'

    items = await loop.run_in_executor(None, lambda: fetch_new_items(gc, store.seen))
    items = list(reversed(items))   # oldest-first, so arrival_seq ascends with ingestion
    rows = await loop.run_in_executor(None, lambda: parse_many(gc, items))
    added, skipped, errors = store.ingest(items, rows)
    _last_count = count
    err = f' <b style="color:#d62728">{errors} err</b>' if errors else ''
    return f'+{added} new ({skipped} without any metric){err} | count={count}'


async def run():
    global _running
    _running = True
    while _running:
        try:
            msg = await _tick()
        except Exception as e:
            msg = f'<span style="color:#d62728">error: {type(e).__name__}: {e}</span>'
        redraw(msg)
        await asyncio.sleep(int(w_poll.value))
    redraw('stopped')


def _stop(_):
    global _running
    _running = False


b_stop.on_click(_stop)

# Re-running this cell must not leave the previous loop alive: two run() tasks means two
# redraws per tick, racing over the same figure.
try:
    task.cancel()
except NameError:
    pass
task = asyncio.ensure_future(run())
print('polling started -- scroll up for the plot')

polling started -- scroll up for the plot
